In [6]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Import CRUD module
from animal_shelter import AnimalShelter

###########################
# Data Manipulation / Model
###########################

username = "aacuser"
password = "SNHU1234"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# Read all records
df = pd.DataFrame.from_records(db.read({}))

# Drop _id because Dash table cannot display ObjectId
if '_id' in df.columns:
    df.drop(columns=['_id'], inplace=True)

#########################
# Dashboard Layout / View
#########################

app = JupyterDash(__name__)

# Load logo image
image_filename = 'Grazioso Salvare Logo.png'   # change if your file name is different
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

app.layout = html.Div([
    html.Center([
        html.H1('CS 340 Grazioso Salvare Dashboard'),
        html.H4('Fisayo G Konigbagbe'),
        html.Img(
            src='data:image/png;base64,{}'.format(encoded_image.decode()),
            style={'width': '250px'}
        )
    ]),
    
    html.Hr(),

    # Interactive filter options
    html.Div([
        html.Label("Filter by Rescue Type:"),
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': 'All Animals (Reset)', 'value': 'RESET'},
                {'label': 'Water Rescue', 'value': 'WATER'},
                {'label': 'Mountain or Wilderness Rescue', 'value': 'MOUNTAIN'},
                {'label': 'Disaster or Individual Tracking', 'value': 'DISASTER'}
            ],
            value='RESET',
            labelStyle={'display': 'inline-block', 'margin-right': '20px'}
        )
    ]),

    html.Hr(),

    # Interactive data table
    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
        data=df.to_dict('records'),
        page_size=10,
        sort_action='native',
        filter_action='native',
        row_selectable='single',
        selected_rows=[0],
        style_table={'overflowX': 'auto'},
        style_cell={
            'textAlign': 'left',
            'minWidth': '120px',
            'width': '120px',
            'maxWidth': '180px',
            'whiteSpace': 'normal'
        },
        style_header={
            'backgroundColor': 'rgb(230, 230, 230)',
            'fontWeight': 'bold'
        }
    ),

    html.Br(),
    html.Hr(),

    # Graph and map side by side
    html.Div(className='row',
             style={'display': 'flex'},
             children=[
                 html.Div(
                     id='graph-id',
                     className='col s12 m6',
                     style={'width': '50%'}
                 ),
                 html.Div(
                     id='map-id',
                     className='col s12 m6',
                     style={'width': '50%'}
                 )
             ])
])

#############################################
# Interaction Between Components / Controller
#############################################

@app.callback(
    Output('datatable-id', 'data'),
    [Input('filter-type', 'value')]
)
def update_dashboard(filter_type):

    if filter_type == 'WATER':
        query = {
            "animal_type": "Dog",
            "breed": {"$in": [
                "Labrador Retriever Mix",
                "Chesapeake Bay Retriever",
                "Newfoundland"
            ]},
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }

    elif filter_type == 'MOUNTAIN':
        query = {
            "animal_type": "Dog",
            "breed": {"$in": [
                "German Shepherd",
                "Alaskan Malamute",
                "Old English Sheepdog",
                "Siberian Husky",
                "Rottweiler"
            ]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }

    elif filter_type == 'DISASTER':
        query = {
            "animal_type": "Dog",
            "breed": {"$in": [
                "Doberman Pinscher",
                "German Shepherd",
                "Golden Retriever",
                "Bloodhound",
                "Rottweiler"
            ]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300}
        }

    else:
        query = {}

    dff = pd.DataFrame.from_records(db.read(query))

    if '_id' in dff.columns:
        dff.drop(columns=['_id'], inplace=True)

    return dff.to_dict('records')


# Pie chart based on breeds in current table
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")]
)
def update_graphs(viewData):

    if viewData is None or len(viewData) == 0:
        dff = df
    else:
        dff = pd.DataFrame.from_dict(viewData)

    return [
        dcc.Graph(
            figure=px.pie(
                dff,
                names='breed',
                title='Preferred Dog Breeds'
            )
        )
    ]


# Highlight selected columns
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):

    if selected_columns is None:
        return []

    return [{
        'if': {'column_id': i},
        'background_color': '#D2F3FF'
    } for i in selected_columns]

# Update geolocation map
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")]
)
def update_map(viewData, index):

    if viewData is None or len(viewData) == 0:
        dff = df
    else:
        dff = pd.DataFrame.from_dict(viewData)

    if index is None or len(index) == 0:
        row = 0
    else:
        row = index[0]

    # Safety check
    if dff.empty:
        return [html.H4("No data to display on map")]

    return [
        dl.Map(
            style={'width': '1000px', 'height': '500px'},
            center=[30.75, -97.48],
            zoom=10,
            children=[
                dl.TileLayer(id="base-layer-id"),
                dl.Marker(
                    position=[dff.iloc[row]['location_lat'], dff.iloc[row]['location_long']],
                    children=[
                        dl.Tooltip(str(dff.iloc[row]['breed'])),
                        dl.Popup([
                            html.H1("Animal Name"),
                            html.P(str(dff.iloc[row]['name']))
                        ])
                    ]
                )
            ]
        )
    ]

# Run app
app.run_server()

Dash app running on https://avenuescreen-tractorclub-3000.codio.io/proxy/8050/
